# Data Cleaning Pipeline
This notebook contains functions for data cleaning including MICE imputation, duplicate removal, NA handling, outlier detection, and fuzzy category condensation.

In [8]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from thefuzz import process, fuzz
import os

# Data Cleaning

## impute_mice
Applies MICE (Multivariate Imputation by Chained Equations) to fill missing values.

In [9]:
def impute_mice(df, columns):
    """
    Applies MICE (Multivariate Imputation by Chained Equations) to fill missing values.
    
    Args:
        df (pd.DataFrame): The input dataframe.
        columns (list): List of column names to apply imputation on.
        
    Returns:
        pd.DataFrame: Dataframe with missing values imputed in specified columns.
    """
    # MICE requires numeric data. Ensure columns are numeric.
    # We create a copy to avoid SettingWithCopy warnings on the original df if it's a slice
    df_imputed = df.copy()
    
    if not columns:
        return df_imputed
        
    imputer = IterativeImputer(random_state=0)
    
    # Fit and transform the selected columns
    # Note: IterativeImputer works on the matrix provided.
    # If columns have correlations with other columns NOT in 'columns', 
    # it might be better to include them in the fit but only update the target columns.
    # For simplicity here, we impute based on the subset.
    imputed_data = imputer.fit_transform(df_imputed[columns])
    
    df_imputed[columns] = imputed_data
    
    return df_imputed

## drop_duplicates_data
Drops duplicate rows from the dataframe.

In [10]:
def drop_duplicates_data(df):
    """
    Drops duplicate rows from the dataframe.
    
    Args:
        df (pd.DataFrame): The input dataframe.
        
    Returns:
        pd.DataFrame: Dataframe without duplicate rows.
    """
    return df.drop_duplicates()

## drop_na_values
Drops rows containing missing values.

In [11]:
def drop_na_values(df, columns=None):
    """
    Drops rows with NA values.
    
    Args:
        df (pd.DataFrame): The input dataframe.
        columns (list, optional): Specific columns to check for NA. Defaults to None (check all).
        
    Returns:
        pd.DataFrame: Dataframe with NAs dropped.
    """
    if columns:
        return df.dropna(subset=columns)
    return df.dropna()

## exclude_outliers_iqr
Excludes outliers based on IQR formula.

In [12]:
def exclude_outliers_iqr(df, column):
    """
    Excludes outliers based on IQR formula.
    
    Args:
        df (pd.DataFrame): Input dataframe.
        column (str): The column to check for outliers.
        
    Returns:
        pd.DataFrame: Dataframe with outliers removed.
    """
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

## condense_categories
Finds categories that are similar at 80% similarity and condenses them into the first value.

In [16]:
def condense_categories(df, column, threshold=80):
    """
    Finds categories that are similar at >= 80% similarity and condenses them 
    into the first value of related similar values.
    
    Args:
        df (pd.DataFrame): Input dataframe.
        column (str): The categorical column to process.
        threshold (int): Similarity score threshold (0-100). Defaults to 80.
        
    Returns:
        pd.DataFrame: Dataframe with condensed categories.
    """
    df_out = df.copy()
    
    # Get unique values, dropping NAs
    unique_values = df_out[column].dropna().unique().tolist()
    
    # Map to store replacements: {original_value: replacement_value}
    replacements = {}
    
    # Iterate through unique values to find clusters
    # We sort to ensure deterministic behavior (e.g. alphabetical preference if needed, 
    # or just processing order). 'First value' depends on order.
    unique_values.sort()
    
    processed = set()
    
    for val in unique_values:
        if val in processed:
            continue
            
        # Find matches for the current value
        # process.extract returns list of tuples (match, score)
        matches = process.extract(val, unique_values, limit=None, scorer=fuzz.token_sort_ratio)
        
        # Filter by threshold
        similar_values = [match[0] for match in matches if match[1] >= threshold]
        
        # The 'first' value is 'val' (since we are iterating)
        # or we could pick the most frequent one if we calculated frequency.
        # The prompt says "Condense into the first value of related similiar values".
        # We'll use 'val' as the canonical form for all similar_values.
        
        for similar in similar_values:
            if similar not in processed:
                replacements[similar] = val
                processed.add(similar)
                
    # Apply replacements
    df_out[column] = df_out[column].replace(replacements)
    
    return df_out

# Data Run

In [ ]:
# Load Data
raw_data_path = '../../data/raw/raw_test_call_data.csv'

# Check if file exists to avoid errors if running blindly
if os.path.exists(raw_data_path):
    df = pd.read_csv(raw_data_path)
    print("Data loaded successfully.")
    print(df.head())
    
    # --- Testing Functions (Commented Out) ---
    
    # 1. Drop Duplicates
    # df_deduped = drop_duplicates_data(df)
    
    # 2. Drop NA (example on specific column)
    # df_no_na = drop_na_values(df, columns=['call_start_time'])
    
    # 3. MICE Imputation
    # Note: MICE works on numeric columns. 'call_duration' is numeric.
    # df_imputed = impute_mice(df, columns=['call_duration'])
    
    # 4. Exclude Outliers (IQR)
    # df_no_outliers = exclude_outliers_iqr(df, 'call_duration')
    
    # 5. Fuzzy Matching Condensation
    # 'disposition' seems to have variations like 'Answered', 'ANSWERED', 'answered'
    #df_condensed = condense_categories(df, 'disposition', threshold=80)
    
    #print(df_condensed['disposition'].unique(), "test")
    #print(df_condensed.head(), "test2")
    
else:
    print(f"File not found at {raw_data_path}")

Data loaded successfully.
   call_id      call_start_time  call_duration  zip_code disposition  \
0     1001  2025-01-20 08:30:15            145     19428    Answered   
1     1002  2025-01-20 08:35:10             32      8003      Missed   
2     1003  2025-01-20 08:40:00            -10     19104    ANSWERED   
3     1004  2025-01-20 08:45:22           3200     90210    Answered   
4     1005  2025-01-20 08:50:11             88     19428    answered   

  billable_status  
0            True  
1           False  
2            True  
3            True  
4             NaN  
['ANSWERED' 'Missed' 'Busy' 'Dropped'] test
   call_id      call_start_time  call_duration  zip_code disposition  \
0     1001  2025-01-20 08:30:15            145     19428    ANSWERED   
1     1002  2025-01-20 08:35:10             32      8003      Missed   
2     1003  2025-01-20 08:40:00            -10     19104    ANSWERED   
3     1004  2025-01-20 08:45:22           3200     90210    ANSWERED   
4     1005  2025-